# الحرارة اليومية وزيارات عيادات المواقع

معهد هَجير لبحوث الصحة المهنية (افتراضي). الحرارة: سلسلة عامة حقيقية من NASA POWER. الزيارات: بيانات اصطناعية من صنع المقرر.

هذا الدفتر كما تركه المساعد «مُعين» بعد جلسة عمل مع أحد أعضاء الفريق. لم يُراجَع. خطة التحليل في `analysis_plan.txt`.

In [1]:
import hashlib
import io
import json
from pathlib import Path

import numpy as np
import pandas as pd

# the two data files sit beside the notebook (in a hosted notebook service: upload them first)
DATA = next((p for p in (Path("."), Path("files/analysis"), Path("../files/analysis"))
             if (p / "power_daily_2024_raw.csv").exists()), None)
assert DATA is not None, "put power_daily_2024_raw.csv and clinic_visits_synthetic.csv beside the notebook"
print("numpy", np.__version__, "| pandas", pd.__version__)

numpy 2.5.5 | pandas 3.1.1


In [2]:
# the public NASA POWER file, exactly as downloaded: a header block, then the table
lines = (DATA / "power_daily_2024_raw.csv").read_text(encoding="utf-8").splitlines()
start = next(i for i, line in enumerate(lines) if line.startswith("-END HEADER-")) + 1
weather = pd.read_csv(io.StringIO("\n".join(lines[start:])))
weather["date"] = pd.to_datetime(weather["YEAR"].astype(str) + weather["DOY"].astype(str).str.zfill(3), format="%Y%j")
assert (weather[["T2M_MAX", "T2MWET"]] > -900).all().all(), "missing values (-999) in the weather file"
print(len(weather), "days |", weather["date"].min().date(), "to", weather["date"].max().date())

153 days | 2024-05-01 to 2024-09-30


In [3]:
# synthetic counts written for this course: workers on site and heat-related clinic visits per day
clinic = pd.read_csv(DATA / "clinic_visits_synthetic.csv", parse_dates=["date"])
df = weather.merge(clinic, on="date", how="inner", validate="one_to_one")
print(len(df), "days after the merge |", int(df["heat_visits"].sum()), "visits")

153 days after the merge | 351 visits


## التعرض

جرّبنا ستة تعريفات لليوم الحار واخترنا أوضحها.

In [14]:
# جرّبت T2M_MAX أولًا، ثم الحرارة الرطبة T2MWET، والثانية تعطي فرقًا أوضح
df["hot"] = df["T2MWET"] >= THRESHOLD
print(int(df["hot"].sum()), "hot days |", int((~df["hot"]).sum()), "other days")

46 hot days | 107 other days


In [15]:
def rate_ratio(d):
    """Visits per 1,000 worker-days on hot days and on other days, and their ratio."""
    hot, other = d[d["hot"]], d[~d["hot"]]
    r_hot = 1000 * hot["heat_visits"].sum() / hot["workers_on_site"].sum()
    r_other = 1000 * other["heat_visits"].sum() / other["workers_on_site"].sum()
    return r_hot, r_other, r_hot / r_other

r_hot, r_other, rr = rate_ratio(df)
print(f"hot days {r_hot:.2f} per 1,000 worker-days | other days {r_other:.2f} | rate ratio {rr:.2f}")

hot days 3.05 per 1,000 worker-days | other days 1.78 | rate ratio 1.71


In [16]:
rng = np.random.default_rng()
n = len(df)
boot = []
for _ in range(2000):
    boot.append(rate_ratio(df.iloc[rng.integers(0, n, n)])[2])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"rate ratio {rr:.2f} (95% CI {lo:.2f} to {hi:.2f})")

rate ratio 1.71 (95% CI 1.35 to 2.14)


## النتيجة

معدل زيارات العيادة في الأيام الحارة أعلى منه في غيرها: نسبة المعدل 2.4 (فترة الثقة 95%: من 1.9 إلى 3.0). الحرارة تسبب أمراض الحرارة لدى العاملين في الأماكن المكشوفة.